# PDF QA Bot (RAG) with LangChain + watsonx.ai

This notebook builds a retrieval-augmented QA bot for PDF documents.

Pipeline:
1. Load a PDF into LangChain `Document` objects.
2. Split documents into chunks.
3. Embed chunks with watsonx.ai embeddings.
4. Store embeddings in a Chroma vector database.
5. Retrieve relevant chunks for a query.
6. Answer with a watsonx.ai LLM using `RetrievalQA`.
7. Expose a simple Gradio UI.


## Notebook Setup

Lightweight setup to reduce noisy warnings in notebook output.


In [1]:
import warnings

from weasel.cli import project_update_dvc

warnings.filterwarnings("ignore")

/Users/davidjbrady/venvs/ml_3.11.9_venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


## Environment Variables

This notebook expects watsonx.ai credentials in a `.env` file (or your shell environment):
- `WATSONX_API_KEY`
- `WATSONX_PROJECT_ID`
- `WATSONX_URL` (for example: `https://us-south.ml.cloud.ibm.com`)

Note: the current cell also checks `OPENAI_API_KEY`. If you are only using watsonx.ai, you can ignore the OpenAI key requirement later by removing that check.


In [2]:
import os
import sys
import subprocess
import importlib.util
import IPython

print("Kernel Python:", sys.executable)

# ------------------------------------------------------------
# 0) Load environment (.env) if python-dotenv is present
# ------------------------------------------------------------
def _load_env_if_possible():
    if importlib.util.find_spec("dotenv") is not None:
        from dotenv import load_dotenv
        load_dotenv(override=True)

_load_env_if_possible()

api_key = os.getenv("OPENAI_API_KEY")
watsonx_api_key = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_url = os.getenv("WATSONX_URL")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set.\n"
        "Create a .env file with:\n"
        "  OPENAI_API_KEY=sk-...\n"
        "and ensure it is loaded before running the notebook."
    )

print("OPENAI_API_KEY loaded ✅ (prefix:", api_key[:6], "…)")
print("WATSONX_API_KEY loaded ✅ (prefix:", watsonx_api_key[:6], "…)")
print("WATSONX PROJECT ID loaded ✅", project_id)
print("WATSONX URL loaded ✅", watsonx_url)

Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
OPENAI_API_KEY loaded ✅ (prefix: sk-pro …)
WATSONX_API_KEY loaded ✅ (prefix: jS9LCS …)
WATSONX PROJECT ID loaded ✅ 9a16631d-bb24-49ad-ac76-768229034001
WATSONX URL loaded ✅ https://us-south.ml.cloud.ibm.com


## Imports

Core libraries used:
- `langchain_*` for document loading, splitting, retrieval, and QA chaining
- `langchain_ibm` + `ibm_watsonx_ai` for watsonx.ai embeddings and LLM inference
- `Chroma` for the local vector database
- `gradio` for the UI


In [3]:
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from ibm_watsonx_ai import Credentials
from langchain_ibm import WatsonxLLM, WatsonxEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA
from huggingface_hub import HfFolder

import gradio as gr

## Step 1: Load PDF Documents

Use `PyPDFLoader` to read a PDF into a list of LangChain `Document` objects.

The loader accepts either:
- A string file path (used by Gradio when `type="filepath"`)
- An object with a `.name` attribute


In [4]:
## Document loader
def document_loader(file):
    file_path = file if isinstance(file, str) else file.name
    loader = PyPDFLoader(file_path)
    loaded_document = loader.load()
    return loaded_document

## Step 2: Split Documents into Chunks

Split long text into smaller overlapping chunks for embedding + retrieval.

- `chunk_size`: target chunk length
- `chunk_overlap`: overlap between adjacent chunks to preserve context


In [5]:
## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

## Step 3: Embedding Model (watsonx.ai)

Convert text chunks into vectors using IBM's Slate 125M English embeddings model:
- `ibm/slate-125m-english-rtrvr-v2`

These vectors are what the vector database indexes for similarity search.


In [6]:
## Embedding model
def watsonx_embedding():
    embed_params = {
        EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
        EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
    }
    watsonx_embedding = WatsonxEmbeddings(
        model_id="ibm/slate-125m-english-rtrvr-v2",
        url=watsonx_url,
        project_id=project_id,
        params=embed_params,
    )
    return watsonx_embedding

## Step 4: Vector Database (Chroma)

Embed all chunks and store them in a Chroma vector store.

In this notebook, the vector store is created from the provided chunks (an in-memory store unless you configure persistence).


In [7]:
## Vector db
def vector_database(chunks):
    embedding_model = watsonx_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb

## Step 5: Retriever

A retriever wraps the vector database and performs similarity search over embeddings to fetch the most relevant chunks for a query.

Default behavior returns the top results by similarity (you can later customize with `search_kwargs`, for example `k`).


In [8]:
## Retriever
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever

## Step 6a: LLM (watsonx.ai)

Build a watsonx.ai LLM wrapper using `ModelInference` and LangChain's `WatsonxLLM`.

Default model:
- `ibm/granite-3-8b-instruct`

Tune generation with parameters like `MAX_NEW_TOKENS` and `TEMPERATURE`.


In [9]:
## LLM
def get_llm(model_id="ibm/granite-3-8b-instruct"):
    params = {
        GenParams.MAX_NEW_TOKENS: 256,
        GenParams.TEMPERATURE: 0.5,
    }
    credentials = {
        "url": watsonx_url,
        "api_key": watsonx_api_key,
    }
    if not credentials["api_key"] or not credentials["url"] or not project_id:
        raise RuntimeError("Missing WATSONX_URL/WATSONX_API_KEY/WATSONX_PROJECT_ID in environment/.env")
    model = ModelInference(
        model_id=model_id,
        params=params,
        credentials=credentials,
        project_id=project_id,
    )
    return WatsonxLLM(watsonx_model=model)

## Step 6b: QA Chain (RetrievalQA)

`RetrievalQA` combines retrieval + generation:
1. Retrieve relevant chunks from the vector store.
2. Provide those chunks as context to the LLM.
3. Generate a final answer.

Here we use `chain_type="stuff"` (simple, concatenates retrieved documents into the prompt) and return source documents for debugging.


In [10]:
## QA Chain
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)
    qa = RetrievalQA.from_chain_type(llm=llm,
                                     chain_type="stuff",
                                     retriever=retriever_obj,
                                     return_source_documents=True)
    response = qa.invoke(query)
    return response['result']

## Step 7: Gradio UI

A minimal UI:
- Upload a single PDF
- Enter a natural-language question
- Display the generated answer

The Gradio `File` component is configured with `type="filepath"` so the function receives a string path.


In [11]:
# Create Gradio interface
# Create Gradio interface
rag_application = gr.Interface(
    fn=retriever_qa,
    #allow_flagging="never",
    inputs=[
        gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),  # Drag and drop file upload
        gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
    ],
    outputs=gr.Textbox(label="Output"),
    title="PDF QA Bot (RAG)",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)


## Launch

Launch the Gradio app locally.

- `debug=True` prints detailed exceptions in the notebook output.
- `show_error=True` (if supported by your Gradio version) also shows the traceback in the UI.


In [ ]:
# Launch the app
launch_kwargs = dict(server_name="localhost", server_port=6868, debug=True)
try:
    rag_application.launch(show_error=True, **launch_kwargs)
except TypeError:
    # Fallback for older Gradio versions that don't support `show_error`.
    rag_application.launch(**launch_kwargs)

* Running on local URL:  http://localhost:6868
* To create a public link, set `share=True` in `launch()`.
